# Homework 2 — Data Cleaning and Validation

This notebook inspects, diagnoses, cleans, and validates the `ohlc_hw`
market-data table. The original table remains unchanged.

## 1. Load and inspect the data

In [1]:
from pathlib import Path
import sqlite3

import numpy as np
import pandas as pd
from IPython.display import display

DB_PATH = Path("data/data.db")

if not DB_PATH.is_file():
    raise FileNotFoundError(f"Database not found: {DB_PATH.resolve()}")

print(f"Using database: {DB_PATH.resolve()}")

Using database: /Users/hb525/Desktop/MFE PreProgram/UCB-MFE-python-preprogram-2026-winter/Homeworks/HW2/Hiba_Lina_Ben_Khaddah/data/data.db


In [2]:
with sqlite3.connect(DB_PATH) as connection:
    tables = pd.read_sql_query(
        """
        SELECT name
        FROM sqlite_master
        WHERE type = 'table'
        ORDER BY name
        """,
        connection,
    )

    raw = pd.read_sql_query(
        "SELECT * FROM ohlc_hw",
        connection,
    )

    reference = pd.read_sql_query(
        "SELECT * FROM ohlc",
        connection,
    )

display(tables)
print(f"Raw data shape: {raw.shape}")
print(f"Reference data shape: {reference.shape}")

,name
0,market_caps
1,ohlc
2,ohlc_hw
3,ohlc_hw_cleaned


Raw data shape: (397711, 8)
Reference data shape: (368251, 8)


In [3]:
display(raw.head())
raw.info()
display(raw.describe(include="all").T)

,ts,open,high,low,close,adj_close,volume,ticker
0,1975-04-14,5.4300,5.4800,5.3600,5.4400,1.640000,446392,CAT
1,1978-04-05,4.8400,4.8900,4.8100,4.8800,0.656990,355200,CVX
2,2018-05-25,124.5400,124.7600,121.2200,122.1900,92.220000,8936649,CVX
3,1995-12-07,9.9700,9.9700,9.8300,9.9400,5.880000,3954150,HD
4,1981-07-23,0.1038,0.1038,0.1038,0.1038,0.080013,34451232,AAPL


<class 'pandas.DataFrame'>
RangeIndex: 397711 entries, 0 to 397710
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   ts         397711 non-null  str    
 1   open       397711 non-null  float64
 2   high       397711 non-null  float64
 3   low        397711 non-null  float64
 4   close      397711 non-null  float64
 5   adj_close  377856 non-null  float64
 6   volume     397711 non-null  int64  
 7   ticker     397711 non-null  str    
dtypes: float64(5), int64(1), str(2)
memory usage: 31.6 MB


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
ts,397711,15868,2020-11-24,40,NaN,NaN,NaN,NaN,NaN,NaN,NaN
open,397711.0,NaN,NaN,NaN,46.285479,67.132594,0.0,4.209938,19.85,57.53,604.26
high,397711.0,NaN,NaN,NaN,46.74766,67.77821,0.005208,4.24,20.06,58.13,608.63
low,397711.0,NaN,NaN,NaN,45.816279,66.477178,0.004801,4.16,19.63,56.95,597.88
close,397711.0,NaN,NaN,NaN,46.292903,67.141879,0.005208,4.2,19.86,57.53,605.4
adj_close,377856.0,NaN,NaN,NaN,51.65225,194.584045,0.002945,1.39,11.96,43.77,10442.510783
volume,397711.0,NaN,NaN,NaN,30977733.689946,126623650.175253,0.0,1730700.0,4648200.0,11023796.5,9276606399.0
ticker,397711,150,PG,15450,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Column meanings

| Column | Meaning |
|---|---|
| `ts` | Observation date |
| `ticker` | Vendor-provided ticker label |
| `open`, `high`, `low`, `close` | Raw daily prices |
| `adj_close` | Vendor-adjusted closing price |
| `volume` | Number of shares traded |

In [4]:
parsed_dates = pd.to_datetime(raw["ts"], errors="coerce")

print("Date range:")
print(parsed_dates.min(), "to", parsed_dates.max())

print("\nNumber of raw ticker labels:")
print(raw["ticker"].nunique(dropna=True))

print("\nExample raw ticker labels:")
display(
    raw["ticker"]
    .drop_duplicates()
    .head(10)
    .rename("raw_ticker")
    .to_frame()
)

print("\nMissing values by column:")
display(raw.isna().sum().rename("missing_count").to_frame())

Date range:
1962-01-02 00:00:00 to 2024-12-06 00:00:00

Number of raw ticker labels:


150

Example raw ticker labels:


,raw_ticker
0,CAT
1,CVX
3,HD
4,AAPL
5,TRV
6,JNJ
8,DIS
9,"<a target=""_blank""><span style=""color: blue"">C..."
10,SHW
11,KO



Missing values by column:


,missing_count
ts,0
open,0
high,0
low,0
close,0
adj_close,19855
volume,0
ticker,0


### Initial observations

- The dataset contains 397,711 rows covering 1962–2024.
- There are 150 raw ticker labels because HTML formatting creates multiple
  versions of the same 30 stocks. Tickers should be normalized before grouping.
- `adj_close` has 19,855 missing values.
- Some prices and volumes are zero, so we need validity checks.
- The rows are not sorted by ticker and date.

## 2. Normalize ticker labels

We should normalize Ticker labels before grouping, checking duplicates, or
calculating returns. I will keep the original labels for auditing.

In [5]:
working = raw.copy()

working["original_ticker"] = working["ticker"]

working["ticker"] = (
    working["ticker"]
    .astype("string")
    .str.replace(r"<[^>]+>", "", regex=True)
    .str.replace(r"\s+", "", regex=True)
    .str.upper()
)

working["ts"] = pd.to_datetime(working["ts"], errors="coerce")

working["ticker_was_normalized"] = (
    working["ticker"] != working["original_ticker"].astype("string")
)

working = (
    working
    .sort_values(["ticker", "ts"])
    .reset_index(drop=True)
)

print("Raw ticker labels:", raw["ticker"].nunique())
print("Normalized tickers:", working["ticker"].nunique())
print(sorted(working["ticker"].dropna().unique()))

Raw ticker labels: 150
Normalized tickers: 30
['AAPL', 'AMGN', 'AMZN', 'AXP', 'BA', 'CAT', 'CRM', 'CSCO', 'CVX', 'DIS', 'GS', 'HD', 'HON', 'IBM', 'JNJ', 'JPM', 'KO', 'MCD', 'MMM', 'MRK', 'MSFT', 'NKE', 'NVDA', 'PG', 'SHW', 'TRV', 'UNH', 'V', 'VZ', 'WMT']


HTML markup, whitespace, and case differences were removed, reducing 150 labels to 30 tickers. Original labels were preserved and rows were sorted by ticker and date before later grouping operations.

## 3. Check duplicate and conflicting records

After ticker normalization, an observation should be uniquely identified by
`(ticker, ts)`. Exact duplicate market records can be removed, but conflicting
records with the same key must be investigated rather than arbitrarily selected.

In [6]:
key_columns = ["ticker", "ts"]

# Audit columns are excluded when deciding whether records are identical.
record_columns = [
    "ticker",
    "ts",
    "open",
    "high",
    "low",
    "close",
    "adj_close",
    "volume",
]

# Preserve every original-to-normalized ticker mapping in an audit table.
ticker_normalization_audit = (
    working[["original_ticker", "ticker"]]
    .drop_duplicates()
    .sort_values(["ticker", "original_ticker"])
    .reset_index(drop=True)
)


all_exact_duplicates = working.duplicated(
    subset=record_columns,
    keep=False,  # Select every copy when displaying examples.
)

display(
    working.loc[all_exact_duplicates, record_columns]
    .sort_values(key_columns)
    .head(10)
)

# Keep one copy of each otherwise identical market record.
extra_exact_duplicates = working.duplicated(
    subset=record_columns,
    keep="first",
)

print("Exact duplicate rows to remove:", extra_exact_duplicates.sum())

deduplicated = (
    working.loc[~extra_exact_duplicates]
    .copy()
    .reset_index(drop=True)
)

# Any repeated key remaining now has conflicting market values.
conflicting_mask = deduplicated.duplicated(
    subset=key_columns,
    keep=False,
)

conflicting_records = (
    deduplicated.loc[conflicting_mask]
    .sort_values(key_columns)
)

print("Rows after exact-duplicate removal:", len(deduplicated))
print("Conflicting rows:", len(conflicting_records))
print(
    "Conflicting ticker/date keys:",
    conflicting_records[key_columns].drop_duplicates().shape[0],
)

display(conflicting_records.head(10))

,ticker,ts,open,high,low,close,adj_close,volume
13,AAPL,1981-01-02,0.15402,0.15402,0.15402,0.15402,0.118730,21660820
14,AAPL,1981-01-02,0.15402,0.15402,0.15402,0.15402,0.118730,21660820
15,AAPL,1981-01-05,0.15067,0.15067,0.15067,0.15067,0.116150,35728032
16,AAPL,1981-01-05,0.15067,0.15067,0.15067,0.15067,0.116150,35728032
42,AAPL,1981-02-10,0.12165,0.12165,0.12165,0.12165,0.093779,18345616
43,AAPL,1981-02-10,0.12165,0.12165,0.12165,0.12165,0.093779,18345616
55,AAPL,1981-02-27,0.11830,0.11830,0.11830,0.11830,0.091197,14761612
56,AAPL,1981-02-27,0.11830,0.11830,0.11830,0.11830,0.091197,14761612
60,AAPL,1981-03-05,0.11551,0.11551,0.11551,0.11551,0.089046,5376004
61,AAPL,1981-03-05,0.11551,0.11551,0.11551,0.11551,0.089046,5376004


Exact duplicate rows to remove: 29460


Rows after exact-duplicate removal: 368251
Conflicting rows: 0
Conflicting ticker/date keys: 0


,ts,open,high,low,close,adj_close,volume,ticker,original_ticker,ticker_was_normalized


In [7]:
working = deduplicated.copy()
# Confirm that exact duplicates and conflicting keys are gone.
assert not working.duplicated(record_columns).any()
assert not working.duplicated(key_columns).any()

print("Duplicate checks passed.")

Duplicate checks passed.


### Duplicate-check result

After ticker normalization, 29,460 exact duplicate rows were removed (
368,251 observations left). No conflicting `(ticker, ts)` keys remained, so we did not need to choose between different prices for the same stock and date. Original ticker mappings remain available in `ticker_normalization_audit`.

## 4. Detect missing data

Missing cells and missing trading-date rows are different problems. Missing
cells are detected with `isna()`. Missing dates require comparison with the
supplied `ohlc` reference calendar.

In [8]:
market_columns = [
    "ts",
    "ticker",
    "open",
    "high",
    "low",
    "close",
    "adj_close",
    "volume",
]

missing_cell_summary = (
    working[market_columns]
    .isna()
    .sum()
    .rename("missing_count")
    .to_frame()
)

display(missing_cell_summary)

,missing_count
ts,0
ticker,0
open,0
high,0
low,0
close,0
adj_close,18374
volume,0


In [9]:
# Prepare the supplied reference dates using the same ticker normalization.
reference_calendar = reference[["ticker", "ts"]].copy()

reference_calendar["ticker"] = (
    reference_calendar["ticker"]
    .astype("string")
    .str.replace(r"<[^>]+>", "", regex=True)
    .str.replace(r"\s+", "", regex=True)
    .str.upper()
)

reference_calendar["ts"] = pd.to_datetime(
    reference_calendar["ts"],
    errors="coerce",
)

reference_calendar = (
    reference_calendar
    .dropna(subset=["ticker", "ts"])
    .drop_duplicates(["ticker", "ts"])
)

In [10]:
def find_missing_trading_days(frame, calendar):
    """Compare each ticker's dates with its supplied reference calendar."""
    summaries = []
    issue_rows = []

    for ticker, ticker_data in frame.groupby("ticker"):
        start_date = ticker_data["ts"].min()
        end_date = ticker_data["ts"].max()

        # Restrict the reference calendar to this ticker's observed history.
        expected_dates = pd.DatetimeIndex(
            calendar.loc[
                (calendar["ticker"] == ticker)
                & calendar["ts"].between(start_date, end_date),
                "ts",
            ].unique()
        )

        observed_dates = pd.DatetimeIndex(
            ticker_data["ts"].dropna().unique()
        )

        missing_dates = expected_dates.difference(
            observed_dates
        ).sort_values()

        day_word = "day" if len(missing_dates) == 1 else "days"

        summaries.append(
            {
                "ticker": ticker,
                "missing_days": len(missing_dates),
                "start_date": start_date,
                "end_date": end_date,
                "summary": (
                    f"Ticker {ticker} is missing {len(missing_dates)} "
                    f"trading {day_word} between {start_date.date()} "
                    f"and {end_date.date()}."
                ),
            }
        )

        # Keep one quality-report row for every absent date.
        for missing_date in missing_dates:
            issue_rows.append(
                {
                    "ticker": ticker,
                    "issue_type": "missing_trading_day",
                    "issue_date": missing_date,
                    "description": (
                        "Date appears in the supplied reference calendar "
                        "but is absent from the homework data."
                    ),
                    "severity": "medium",
                }
            )

    issue_columns = [
        "ticker",
        "issue_type",
        "issue_date",
        "description",
        "severity",
    ]

    return (
        pd.DataFrame(summaries),
        pd.DataFrame(issue_rows, columns=issue_columns),
    )

In [11]:
demo_calendar = pd.DataFrame(
    {
        "ticker": ["DEMO"] * 4,
        "ts": pd.to_datetime(
            [
                "2024-01-02",
                "2024-01-03",
                "2024-01-04",
                "2024-01-05",
            ]
        ),
    }
)

# Deliberately remove January 3 to test the detector.
demo_data = demo_calendar.drop(index=1)

demo_summary, demo_missing_issues = find_missing_trading_days(
    demo_data,
    demo_calendar,
)

display(demo_summary)
display(demo_missing_issues)

assert demo_missing_issues["issue_date"].tolist() == [
    pd.Timestamp("2024-01-03")
]

,ticker,missing_days,start_date,end_date,summary
0,DEMO,1,2024-01-02,2024-01-05,Ticker DEMO is missing 1 trading day between 2...


,ticker,issue_type,issue_date,description,severity
0,DEMO,missing_trading_day,2024-01-03,Date appears in the supplied reference calenda...,medium


In [12]:
missing_date_summary, missing_date_issues = (
    find_missing_trading_days(
        working,
        reference_calendar,
    )
)

display(missing_date_summary)
print("Total missing trading dates:", len(missing_date_issues))

,ticker,missing_days,start_date,end_date,summary
0,AAPL,0,1980-12-12,2024-10-18,Ticker AAPL is missing 0 trading days between ...
1,AMGN,0,1983-06-17,2024-10-18,Ticker AMGN is missing 0 trading days between ...
2,AMZN,0,1997-05-15,2024-12-06,Ticker AMZN is missing 0 trading days between ...
3,AXP,0,1972-04-03,2024-10-18,Ticker AXP is missing 0 trading days between 1...
4,BA,0,1962-01-02,2024-10-18,Ticker BA is missing 0 trading days between 19...
5,CAT,0,1962-01-02,2024-10-18,Ticker CAT is missing 0 trading days between 1...
6,CRM,0,2004-06-23,2024-10-18,Ticker CRM is missing 0 trading days between 2...
7,CSCO,0,1990-02-16,2024-10-18,Ticker CSCO is missing 0 trading days between ...
8,CVX,0,1962-01-02,2024-12-06,Ticker CVX is missing 0 trading days between 1...
9,DIS,0,1962-01-02,2024-12-06,Ticker DIS is missing 0 trading days between 1...


Total missing trading dates: 0


### Missing-data result

After deduplication, `adj_close` contains 18,374 missing cells; the other market
columns contain no missing cells. No trading-date rows are missing relative to
each ticker's supplied `ohlc` calendar. The detector was also verified using a
small example with one deliberately removed date.

## 5. Diagnose prices, volume, and returns

These checks diagnose the data before any repairs. Extreme returns are review
flags, not automatic evidence of incorrect prices.

In [13]:
raw_price_columns = ["open", "high", "low", "close"]
price_columns = raw_price_columns + ["adj_close"]

diagnostic = working.copy()

# Exclude invalid closes before calculating returns.
valid_close = diagnostic["close"].where(
    np.isfinite(diagnostic["close"])
    & diagnostic["close"].gt(0)
)

diagnostic["simple_return"] = (
    valid_close
    .groupby(diagnostic["ticker"])
    .pct_change(fill_method=None)
)

# Calculate each ticker's mean and sample standard deviation.
mean_return = (
    diagnostic
    .groupby("ticker")["simple_return"]
    .transform("mean")
)

std_return = (
    diagnostic
    .groupby("ticker")["simple_return"]
    .transform(lambda values: values.std(ddof=1))
)

# Undefined returns and zero/undefined standard deviations cannot be flagged.
usable_return_statistics = (
    diagnostic["simple_return"].notna()
    & mean_return.notna()
    & std_return.notna()
    & np.isfinite(mean_return)
    & np.isfinite(std_return)
    & std_return.gt(0)
)

extreme_return_mask = (
    usable_return_statistics
    & (
        diagnostic["simple_return"] - mean_return
    ).abs().gt(3 * std_return)
)

In [14]:
# Add validation masks for each issue type.
finite_raw_prices = np.isfinite(
    diagnostic[raw_price_columns]
).all(axis=1)

issue_masks = {
    "invalid_raw_price": (
        (~np.isfinite(diagnostic[raw_price_columns]))
        | diagnostic[raw_price_columns].le(0)
    ).any(axis=1),

    "invalid_adjusted_close": (
        diagnostic["adj_close"].isna()
        | ~np.isfinite(diagnostic["adj_close"])
        | diagnostic["adj_close"].le(0)
    ),

    "high_below_close": (
        finite_raw_prices
        & diagnostic["high"].lt(diagnostic["close"])
    ),

    "low_above_close": (
        finite_raw_prices
        & diagnostic["low"].gt(diagnostic["close"])
    ),

    "open_outside_bounds": (
        finite_raw_prices
        & ~diagnostic["open"].between(
            diagnostic["low"],
            diagnostic["high"],
        )
    ),

    "low_above_high": (
        finite_raw_prices
        & diagnostic["low"].gt(diagnostic["high"])
    ),

    "zero_volume": diagnostic["volume"].eq(0),

    "invalid_volume": (
        diagnostic["volume"].isna()
        | ~np.isfinite(diagnostic["volume"])
        | diagnostic["volume"].lt(0)
    ),

    "extreme_return": extreme_return_mask,
}

issue_counts = pd.Series(
    {
        issue_type: int(mask.sum())
        for issue_type, mask in issue_masks.items()
    },
    name="issue_count",
)

display(issue_counts.to_frame())

,issue_count
invalid_raw_price,5
invalid_adjusted_close,18374
high_below_close,0
low_above_close,0
open_outside_bounds,5
low_above_high,0
zero_volume,75
invalid_volume,0
extreme_return,5056


### Diagnostic result

The checks found five invalid raw-price rows, 18,374 invalid or missing adjusted
closes, five invalid opening-price bounds, and 75 zero-volume observations.
The return rule flagged 5,056 observations for review. These return flags are
not removed automatically because large market moves may be valid.

## 6. Create the Data Quality Report

High-severity issues violate required price, volume, or OHLC rules. Medium
severity means an observation requires investigation, low severity points at
a condition such as zero volume that may be valid but should be reviewed.
A row may appear multiple times when it has multiple issues.

In [15]:
issue_metadata = {
    "invalid_raw_price": (
        "At least one raw OHLC price is missing, non-finite, or nonpositive.",
        "high",
    ),
    "invalid_adjusted_close": (
        "Adjusted close is missing, non-finite, or nonpositive.",
        "high",
    ),
    "high_below_close": (
        "Daily high is below the closing price.",
        "high",
    ),
    "low_above_close": (
        "Daily low is above the closing price.",
        "high",
    ),
    "open_outside_bounds": (
        "Opening price is outside the daily low-high range.",
        "high",
    ),
    "low_above_high": (
        "Daily low is above the daily high.",
        "high",
    ),
    "zero_volume": (
        "Zero volume may indicate stale data.",
        "low",
    ),
    "invalid_volume": (
        "Volume is missing, negative, or non-finite.",
        "high",
    ),
    "extreme_return": (
        "Raw-close return differs from its ticker mean by more than "
        "three sample standard deviations.",
        "medium",
    ),
}

In [16]:
def build_issue_report(frame, masks, metadata):
    """Convert diagnostic masks into the required issue-report format."""
    report_parts = []

    for issue_type, mask in masks.items():
        description, severity = metadata[issue_type]

        issue_rows = (
            frame.loc[mask, ["ticker", "ts"]]
            .rename(columns={"ts": "issue_date"})
            .copy()
        )

        issue_rows["issue_type"] = issue_type
        issue_rows["description"] = description
        issue_rows["severity"] = severity

        report_parts.append(issue_rows)

    report_columns = [
        "ticker",
        "issue_type",
        "issue_date",
        "description",
        "severity",
    ]

    return pd.concat(
        report_parts,
        ignore_index=True,
    )[report_columns]

In [17]:
market_quality_issues = build_issue_report(
    diagnostic,
    issue_masks,
    issue_metadata,
)

# Missing dates already have the same report columns.
quality_report_before = pd.concat(
    [
        market_quality_issues,
        missing_date_issues,
    ],
    ignore_index=True,
)

quality_report_before = (
    quality_report_before
    .sort_values(["ticker", "issue_date", "issue_type"])
    .reset_index(drop=True)
)

display(quality_report_before.head(20))

display(
    quality_report_before
    .groupby(["issue_type", "severity"])
    .size()
    .rename("issue_count")
    .to_frame()
)

print("Total issue rows:", len(quality_report_before))

,ticker,issue_type,issue_date,description,severity
0,AAPL,extreme_return,1980-12-26 00:00:00,Raw-close return differs from its ticker mean ...,medium
1,AAPL,invalid_adjusted_close,1981-01-08 00:00:00,"Adjusted close is missing, non-finite, or nonp...",high
2,AAPL,invalid_adjusted_close,1981-01-13 00:00:00,"Adjusted close is missing, non-finite, or nonp...",high
3,AAPL,invalid_adjusted_close,1981-02-12 00:00:00,"Adjusted close is missing, non-finite, or nonp...",high
4,AAPL,extreme_return,1981-04-02 00:00:00,Raw-close return differs from its ticker mean ...,medium
5,AAPL,invalid_adjusted_close,1981-04-16 00:00:00,"Adjusted close is missing, non-finite, or nonp...",high
6,AAPL,invalid_adjusted_close,1981-05-19 00:00:00,"Adjusted close is missing, non-finite, or nonp...",high
7,AAPL,invalid_adjusted_close,1981-06-04 00:00:00,"Adjusted close is missing, non-finite, or nonp...",high
8,AAPL,invalid_adjusted_close,1981-07-14 00:00:00,"Adjusted close is missing, non-finite, or nonp...",high
9,AAPL,invalid_adjusted_close,1981-07-20 00:00:00,"Adjusted close is missing, non-finite, or nonp...",high


,,issue_count
issue_type,severity,
extreme_return,medium,5056
invalid_adjusted_close,high,18374
invalid_raw_price,high,5
open_outside_bounds,high,5
zero_volume,low,75


Total issue rows: 23515


### Quality-report result

The report preserves one row per detected issue and allows multiple issues for
the same observation. Invalid prices and relationships are high severity,
extreme returns require investigation, and zero volume is kept as a
low-severity review flag. This report records the diagnosis before repairs.

## 7. Verify the detector with injected errors

Some required issue types do not occur in the course data. A small artificial
example verifies that the same diagnostic rules detect those cases correctly.

In [18]:
def detect_market_issues(frame):
    """Diagnose prices, volume, OHLC relationships, and extreme returns."""
    data = (
        frame
        .sort_values(["ticker", "ts"])
        .copy()
    )

    # Invalid closes cannot produce meaningful returns.
    valid_close = data["close"].where(
        np.isfinite(data["close"])
        & data["close"].gt(0)
    )

    data["simple_return"] = (
        valid_close
        .groupby(data["ticker"])
        .pct_change(fill_method=None)
    )

    grouped_returns = data["simple_return"].groupby(
        data["ticker"]
    )

    mean_return = grouped_returns.transform("mean")
    std_return = grouped_returns.transform(
        lambda values: values.std(ddof=1)
    )

    extreme_return = (
        data["simple_return"].notna()
        & std_return.notna()
        & np.isfinite(std_return)
        & std_return.gt(0)
        & (
            data["simple_return"] - mean_return
        ).abs().gt(3 * std_return)
    )

    finite_raw = np.isfinite(
        data[raw_price_columns]
    ).all(axis=1)

    masks = {
        "invalid_raw_price": (
            (~np.isfinite(data[raw_price_columns]))
            | data[raw_price_columns].le(0)
        ).any(axis=1),

        "invalid_adjusted_close": (
            data["adj_close"].isna()
            | ~np.isfinite(data["adj_close"])
            | data["adj_close"].le(0)
        ),

        "high_below_close": (
            finite_raw
            & data["high"].lt(data["close"])
        ),

        "low_above_close": (
            finite_raw
            & data["low"].gt(data["close"])
        ),

        "open_outside_bounds": (
            finite_raw
            & ~data["open"].between(
                data["low"],
                data["high"],
            )
        ),

        "low_above_high": (
            finite_raw
            & data["low"].gt(data["high"])
        ),

        "zero_volume": data["volume"].eq(0),

        "invalid_volume": (
            data["volume"].isna()
            | ~np.isfinite(data["volume"])
            | data["volume"].lt(0)
        ),

        "extreme_return": extreme_return,
    }

    report = build_issue_report(
        data,
        masks,
        issue_metadata,
    )

    return data, masks, report

In [19]:
diagnostic_check, masks_check, report_check = (
    detect_market_issues(working)
)

expected_counts = (
    market_quality_issues["issue_type"]
    .value_counts()
    .sort_index()
)

reproduced_counts = (
    report_check["issue_type"]
    .value_counts()
    .sort_index()
)

pd.testing.assert_series_equal(
    reproduced_counts,
    expected_counts,
)

print("Reusable detector reproduces the original issue counts.")

Reusable detector reproduces the original issue counts.


In [20]:
injected_example = pd.DataFrame(
    {
        "ticker": ["DEMO"] * 4,
        "ts": pd.to_datetime(
            [
                "2024-01-02",
                "2024-01-03",
                "2024-01-04",
                "2024-01-05",
            ]
        ),
        "open": [10.0, 10.0, 10.0, np.nan],
        "high": [11.0, 9.5, 11.0, 11.0],
        "low": [9.0, 9.0, 12.0, 9.0],
        "close": [10.0, 10.0, 10.0, 10.0],
        "adj_close": [10.0, 10.0, 10.0, 0.0],
        "volume": [100.0, 100.0, 100.0, -5.0],
    }
)

_, _, injected_report = detect_market_issues(
    injected_example
)

display(injected_example)
display(injected_report)

,ticker,ts,open,high,low,close,adj_close,volume
0,DEMO,2024-01-02,10.0,11.0,9.0,10.0,10.0,100.0
1,DEMO,2024-01-03,10.0,9.5,9.0,10.0,10.0,100.0
2,DEMO,2024-01-04,10.0,11.0,12.0,10.0,10.0,100.0
3,DEMO,2024-01-05,NaN,11.0,9.0,10.0,0.0,-5.0


,ticker,issue_type,issue_date,description,severity
0,DEMO,invalid_raw_price,2024-01-05,"At least one raw OHLC price is missing, non-fi...",high
1,DEMO,invalid_adjusted_close,2024-01-05,"Adjusted close is missing, non-finite, or nonp...",high
2,DEMO,high_below_close,2024-01-03,Daily high is below the closing price.,high
3,DEMO,low_above_close,2024-01-04,Daily low is above the closing price.,high
4,DEMO,open_outside_bounds,2024-01-03,Opening price is outside the daily low-high ra...,high
5,DEMO,open_outside_bounds,2024-01-04,Opening price is outside the daily low-high ra...,high
6,DEMO,low_above_high,2024-01-04,Daily low is above the daily high.,high
7,DEMO,invalid_volume,2024-01-05,"Volume is missing, negative, or non-finite.",high


In [21]:
expected_injected_issues = {
    "high_below_close",
    "low_above_close",
    "low_above_high",
    "open_outside_bounds",
    "invalid_raw_price",
    "invalid_adjusted_close",
    "invalid_volume",
}

assert expected_injected_issues.issubset(
    set(injected_report["issue_type"])
)

assert "extreme_return" not in set(
    injected_report["issue_type"]
)

print("Injected diagnostic checks passed.")

Injected diagnostic checks passed.


### Injected diagnostic result

The injected example confirmed that the detector identifies impossible OHLC
relationships, missing raw prices, invalid adjusted close, and negative volume.
Constant prices produced zero return volatility and correctly generated no
extreme-return flag.

## 8. Handle missing raw prices and volume

We remove rows with all four raw prices missing before filling. Partial raw
price gaps are forward-filled only within a sorted ticker. Missing volume is
replaced with zero and flagged because an unknown value is not evidence of no
trading. We check OHLC bounds again after filling.

In [22]:
def handle_missing_values(frame):
    """Handle missing raw prices and volume while preserving repair flags."""
    data = (
        frame
        .sort_values(["ticker", "ts"])
        .copy()
    )

    # Treat infinite raw prices as unusable before checking missingness.
    data[raw_price_columns] = data[
        raw_price_columns
    ].replace([np.inf, -np.inf], np.nan)

    all_raw_missing = data[
        raw_price_columns
    ].isna().all(axis=1)

    removed_rows = data.loc[all_raw_missing].copy()
    removed_rows["removal_reason"] = (
        "All raw OHLC prices were missing."
    )

    data = data.loc[~all_raw_missing].copy()

    missing_raw_before = data[
        raw_price_columns
    ].isna()

    # Forward filling stays inside each ticker's chronological history.
    filled_raw_prices = (
        data
        .groupby("ticker")[raw_price_columns]
        .ffill()
    )

    filled_cells = (
        missing_raw_before
        & filled_raw_prices.notna()
    )

    data[raw_price_columns] = filled_raw_prices
    data["raw_price_filled"] = filled_cells.any(axis=1)

    # Fill only genuinely missing volume; observed zeros remain unchanged.
    data["volume_filled"] = data["volume"].isna()
    data.loc[data["volume_filled"], "volume"] = 0

    return (
        data.reset_index(drop=True),
        removed_rows.reset_index(drop=True),
    )

In [23]:
cleaned, removed_all_raw_missing = (
    handle_missing_values(working)
)

# These flags will be updated by later repair steps.
cleaned["ohlc_repaired"] = False
cleaned["adj_close_reconstructed"] = False

remaining_raw_gaps = (
    cleaned[raw_price_columns]
    .isna()
    .any(axis=1)
)

print(
    "Rows removed because all raw prices were missing:",
    len(removed_all_raw_missing),
)
print(
    "Rows with at least one forward-filled raw price:",
    cleaned["raw_price_filled"].sum(),
)
print(
    "Rows with missing volume replaced by zero:",
    cleaned["volume_filled"].sum(),
)
print(
    "Rows with unresolved leading raw-price gaps:",
    remaining_raw_gaps.sum(),
)

Rows removed because all raw prices were missing: 0
Rows with at least one forward-filled raw price: 0
Rows with missing volume replaced by zero: 0
Rows with unresolved leading raw-price gaps: 0


### Missing-value repair result

The course data contained no missing raw OHLC or volume cells after
deduplication, so no rows required these repairs. The function still implements
the required behavior and retains explicit repair flags. Missing adjusted close
is handled separately using date-specific reference adjustment factors.

## 9. Repair invalid OHLC prices

When only the opening price is invalid and the same-day high, low, and close
are valid, the open is estimated as their median. For other invalid OHLC rows,
all raw prices are replaced with the previous validated close. Rows without a
defensible repair value are quarantined. Every repair is flagged and all OHLC
bounds are rechecked.

In [24]:
def valid_ohlc_mask(frame):
    """Return True for rows with finite, positive, internally valid OHLC."""
    finite_and_positive = (
        np.isfinite(frame[raw_price_columns]).all(axis=1)
        & frame[raw_price_columns].gt(0).all(axis=1)
    )

    valid_bounds = (
        frame["open"].between(frame["low"], frame["high"])
        & frame["close"].between(frame["low"], frame["high"])
    )

    return finite_and_positive & valid_bounds

In [25]:
def repair_invalid_ohlc(frame):
    """Repair defensible OHLC errors and quarantine unresolved rows."""
    data = (
        frame
        .sort_values(["ticker", "ts"])
        .copy()
    )

    data["ohlc_repaired"] = False
    data["close_repaired"] = False
    data["ohlc_repair_method"] = pd.NA

    # Preserve original values for a separate repair audit.
    original_prices = data[
        key_columns + raw_price_columns
    ].copy()

    initially_valid = valid_ohlc_mask(data)

    # Determine whether high, low, and close are usable without the open.
    valid_hlc = (
        np.isfinite(data[["high", "low", "close"]]).all(axis=1)
        & data[["high", "low", "close"]].gt(0).all(axis=1)
        & data["low"].le(data["high"])
        & data["close"].between(data["low"], data["high"])
    )

    invalid_open = (
        data["open"].isna()
        | ~np.isfinite(data["open"])
        | data["open"].le(0)
        | ~data["open"].between(data["low"], data["high"])
    )

    # Option 3: estimate the invalid open from valid same-day HLC prices.
    repair_open = (
        ~initially_valid
        & valid_hlc
        & invalid_open
    )

    estimated_open = (
        data.loc[
            repair_open,
            ["high", "low", "close"],
        ]
        .median(axis=1)
    )

    data.loc[repair_open, "open"] = estimated_open
    data.loc[repair_open, "ohlc_repaired"] = True
    data.loc[
        repair_open,
        "ohlc_repair_method",
    ] = "open_estimated_from_hlc_median"

    valid_after_open_repair = valid_ohlc_mask(data)
    remaining_invalid = ~valid_after_open_repair

    # Find the latest earlier validated close within each ticker.
    previous_valid_close = (
        data["close"]
        .where(valid_after_open_repair)
        .groupby(data["ticker"])
        .ffill()
    )

    repair_with_previous_close = (
        remaining_invalid
        & previous_valid_close.notna()
    )

    # Option 2: replace all raw prices with the previous validated close.
    for column in raw_price_columns:
        data.loc[
            repair_with_previous_close,
            column,
        ] = previous_valid_close.loc[
            repair_with_previous_close
        ]

    data.loc[
        repair_with_previous_close,
        "ohlc_repaired",
    ] = True
    data.loc[
        repair_with_previous_close,
        "close_repaired",
    ] = True
    data.loc[
        repair_with_previous_close,
        "ohlc_repair_method",
    ] = "all_ohlc_replaced_with_previous_close"

    unresolved = ~valid_ohlc_mask(data)

    quarantine = data.loc[unresolved].copy()
    quarantine["quarantine_reason"] = (
        "Invalid OHLC could not be repaired defensibly."
    )

    repaired = data.loc[~unresolved].copy()

    # Build an audit containing original and repaired prices.
    repaired_indices = repaired.index[
        repaired["ohlc_repaired"]
    ]

    repair_audit = original_prices.loc[
        repaired_indices
    ].copy()

    repair_audit = repair_audit.rename(
        columns={
            column: f"original_{column}"
            for column in raw_price_columns
        }
    )

    for column in raw_price_columns:
        repair_audit[f"repaired_{column}"] = repaired.loc[
            repaired_indices,
            column,
        ]

    repair_audit["repair_method"] = repaired.loc[
        repaired_indices,
        "ohlc_repair_method",
    ]

    return (
        repaired.reset_index(drop=True),
        quarantine.reset_index(drop=True),
        repair_audit.reset_index(drop=True),
    )

In [26]:
cleaned, ohlc_quarantine, ohlc_repair_audit = (
    repair_invalid_ohlc(cleaned)
)

assert valid_ohlc_mask(cleaned).all()

print("Repaired OHLC rows:", cleaned["ohlc_repaired"].sum())
print("Quarantined OHLC rows:", len(ohlc_quarantine))

display(
    cleaned["ohlc_repair_method"]
    .value_counts(dropna=False)
    .rename("row_count")
    .to_frame()
)

display(ohlc_repair_audit)

Repaired OHLC rows: 5
Quarantined OHLC rows: 0


,row_count
ohlc_repair_method,
<NA>,368246
open_estimated_from_hlc_median,5


,ticker,ts,original_open,original_high,original_low,original_close,repaired_open,repaired_high,repaired_low,repaired_close,repair_method
0,CVX,1962-01-02,0.0,3.30000,3.24000,3.30000,3.30000,3.30000,3.24000,3.30000,open_estimated_from_hlc_median
1,JNJ,1962-01-02,0.0,0.22338,0.22222,0.22338,0.22338,0.22338,0.22222,0.22338,open_estimated_from_hlc_median
2,MCD,1966-07-05,0.0,0.27366,0.26749,0.26955,0.26955,0.27366,0.26749,0.26955,open_estimated_from_hlc_median
3,MRK,1962-01-02,0.0,0.39703,0.39317,0.39537,0.39537,0.39703,0.39317,0.39537,open_estimated_from_hlc_median
4,PG,1962-01-02,0.0,1.44000,1.42000,1.43000,1.43000,1.44000,1.42000,1.43000,open_estimated_from_hlc_median


### OHLC-repair result

Five zero opening prices occurred on the first observation for their tickers,
so no previous close was available. Because each row’s high, low, and close
were valid, the opening price was estimated as their median. No rows
required quarantine, and all retained OHLC relationships passed validation.

## 10. Reconstruct adjusted close

Missing, invalid, or inconsistent adjusted closes are reconstructed using an
exact ticker/date adjustment factor from the supplied `ohlc` reference table:

`adjustment factor = reference adjusted close / reference raw close`

`reconstructed adjusted close = validated raw close × adjustment factor`

An observed adjusted close is considered inconsistent when it differs from the
expected value by more than one cent. The factor is never forward-filled because
a corporate action may change it between dates. Rows without an exact usable
factor are quarantined.

In [27]:
def prepare_adjustment_factors(reference_frame):
    """Create unique date-specific adjustment factors from `ohlc`."""
    factors = reference_frame.copy()

    factors["ticker"] = (
        factors["ticker"]
        .astype("string")
        .str.replace(r"<[^>]+>", "", regex=True)
        .str.replace(r"\s+", "", regex=True)
        .str.upper()
    )

    factors["ts"] = pd.to_datetime(
        factors["ts"],
        errors="coerce",
    )

    valid_reference = (
        factors["ticker"].notna()
        & factors["ts"].notna()
        & np.isfinite(factors["close"])
        & factors["close"].gt(0)
        & np.isfinite(factors["adj_close"])
        & factors["adj_close"].gt(0)
    )

    factors = factors.loc[
        valid_reference,
        ["ticker", "ts", "close", "adj_close"],
    ].copy()

    # Conflicting factors for one key would make reconstruction ambiguous.
    assert not factors.duplicated(
        ["ticker", "ts"]
    ).any()

    factors["reference_adjustment_factor"] = (
        factors["adj_close"]
        / factors["close"]
    )

    return factors[
        [
            "ticker",
            "ts",
            "reference_adjustment_factor",
        ]
    ]

In [28]:
adjustment_factors = prepare_adjustment_factors(
    reference
)

display(adjustment_factors.head())
print("Available reference factors:", len(adjustment_factors))

,ticker,ts,reference_adjustment_factor
0,JNJ,2024-10-14,1.0
1,JNJ,2024-10-11,1.0
2,JNJ,2024-10-10,1.0
3,JNJ,2024-09-30,1.0
4,JNJ,2024-09-27,1.0


Available reference factors: 368251


### Detect inconsistent adjusted closes

A positive adjusted close can still be corrupted. We compare each observed
adjusted close with the value implied by the trusted exact ticker/date
adjustment factor. Material inconsistencies are high-severity issues and must
be reconstructed.

In [29]:
adjustment_check = (
    cleaned[
        ["ticker", "ts", "close", "adj_close"]
    ]
    .merge(
        adjustment_factors,
        on=["ticker", "ts"],
        how="left",
        validate="one_to_one",
    )
)

adjustment_check["expected_adj_close"] = (
    adjustment_check["close"]
    * adjustment_check["reference_adjustment_factor"]
)

valid_observed_adj_close = (
    adjustment_check["adj_close"].notna()
    & np.isfinite(adjustment_check["adj_close"])
    & adjustment_check["adj_close"].gt(0)
)

valid_expected_adj_close = (
    adjustment_check["expected_adj_close"].notna()
    & np.isfinite(adjustment_check["expected_adj_close"])
    & adjustment_check["expected_adj_close"].gt(0)
)

inconsistent_adj_close = (
    valid_observed_adj_close
    & valid_expected_adj_close
    & ~np.isclose(
        adjustment_check["adj_close"],
        adjustment_check["expected_adj_close"],
        rtol=0,
        atol=0.01,
    )
)

print(
    "Positive but inconsistent adjusted closes:",
    inconsistent_adj_close.sum(),
)

display(
    adjustment_check.loc[
        inconsistent_adj_close,
        [
            "ticker",
            "ts",
            "close",
            "adj_close",
            "expected_adj_close",
        ],
    ].head(10)
)

Positive but inconsistent adjusted closes: 10417


,ticker,ts,close,adj_close,expected_adj_close
35,AAPL,1981-02-03,0.123330,1.264587,0.095069
93,AAPL,1981-04-28,0.126120,2.036653,0.097220
101,AAPL,1981-05-08,0.125000,2.129290,0.096360
157,AAPL,1981-07-29,0.106030,0.506329,0.081734
163,AAPL,1981-08-06,0.112720,2.009828,0.086896
167,AAPL,1981-08-13,0.103800,1.122224,0.080013
168,AAPL,1981-08-14,0.102120,1.171347,0.078722
206,AAPL,1981-10-08,0.082589,1.245215,0.063666
264,AAPL,1981-12-31,0.098772,1.730921,0.076141
274,AAPL,1982-01-15,0.089286,1.577125,0.068829


In [30]:
inconsistent_adjusted_issues = (
    adjustment_check.loc[
        inconsistent_adj_close,
        ["ticker", "ts"],
    ]
    .rename(columns={"ts": "issue_date"})
    .copy()
)

inconsistent_adjusted_issues["issue_type"] = (
    "inconsistent_adjusted_close"
)
inconsistent_adjusted_issues["description"] = (
    "Adjusted close is inconsistent with the trusted "
    "exact ticker/date adjustment factor."
)
inconsistent_adjusted_issues["severity"] = "high"

# Make the cell safe to rerun without duplicating issue rows.
quality_report_before = quality_report_before.loc[
    quality_report_before["issue_type"]
    != "inconsistent_adjusted_close"
].copy()

quality_report_before = (
    pd.concat(
        [
            quality_report_before,
            inconsistent_adjusted_issues,
        ],
        ignore_index=True,
    )
    .sort_values(
        ["ticker", "issue_date", "issue_type"]
    )
    .reset_index(drop=True)
)

assert len(inconsistent_adjusted_issues) == 10_417

display(
    quality_report_before
    .groupby(["issue_type", "severity"])
    .size()
    .rename("issue_count")
    .to_frame()
)

,,issue_count
issue_type,severity,
extreme_return,medium,5056
inconsistent_adjusted_close,high,10417
invalid_adjusted_close,high,18374
invalid_raw_price,high,5
open_outside_bounds,high,5
zero_volume,low,75


In [31]:
def reconstruct_adjusted_close(frame, factors):
    """Reconstruct adjusted close using exact ticker/date factors."""
    data = frame.merge(
        factors,
        on=["ticker", "ts"],
        how="left",
        validate="one_to_one",
    )

    data["adj_close_reconstructed"] = False
    data["adj_close_repair_method"] = pd.NA

    # Preserve the original value for the repair audit.
    original_adjusted_close = data["adj_close"].copy()

    invalid_adjusted_close = (
        data["adj_close"].isna()
        | ~np.isfinite(data["adj_close"])
        | data["adj_close"].le(0)
    )

    usable_factor = (
        np.isfinite(
            data["reference_adjustment_factor"]
        )
        & data["reference_adjustment_factor"].gt(0)
        & np.isfinite(data["close"])
        & data["close"].gt(0)
    )

    expected_adjusted_close = (
        data["close"]
        * data["reference_adjustment_factor"]
    )

    valid_observed_adjusted_close = (
        data["adj_close"].notna()
        & np.isfinite(data["adj_close"])
        & data["adj_close"].gt(0)
    )

    inconsistent_adjusted_close = (
        valid_observed_adjusted_close
        & usable_factor
        & ~np.isclose(
            data["adj_close"],
            expected_adjusted_close,
            rtol=0,
            atol=0.01,
        )
    )

    # Reconstruct missing, corrupted, or repair-affected values.
    needs_reconstruction = (
        invalid_adjusted_close
        | inconsistent_adjusted_close
        | data["raw_price_filled"]
        | data["close_repaired"]
    )

    reconstruct = (
        needs_reconstruction
        & usable_factor
    )

    data.loc[reconstruct, "adj_close"] = (
        data.loc[reconstruct, "close"]
        * data.loc[
            reconstruct,
            "reference_adjustment_factor",
        ]
    )

    data.loc[
        reconstruct,
        "adj_close_reconstructed",
    ] = True

    data.loc[
        reconstruct,
        "adj_close_repair_method",
    ] = "exact_reference_ticker_date_factor"

    data.loc[
        reconstruct & inconsistent_adjusted_close,
        "adj_close_repair_method",
    ] = (
        "inconsistent_value_rebuilt_with_exact_reference_factor"
    )

    unresolved = (
        needs_reconstruction
        & ~usable_factor
    )

    quarantine = data.loc[unresolved].copy()
    quarantine["quarantine_reason"] = (
        "Adjusted close could not be reconstructed from an "
        "exact ticker/date reference factor."
    )

    retained = data.loc[~unresolved].copy()

    audit = retained.loc[
        retained["adj_close_reconstructed"],
        [
            "ticker",
            "ts",
            "close",
            "reference_adjustment_factor",
            "adj_close",
            "adj_close_repair_method",
        ],
    ].copy()

    audit.insert(
        3,
        "original_adj_close",
        original_adjusted_close.loc[audit.index],
    )

    retained = retained.drop(
        columns="reference_adjustment_factor"
    )

    return (
        retained.reset_index(drop=True),
        quarantine.reset_index(drop=True),
        audit.reset_index(drop=True),
    )

In [32]:
cleaned, adjusted_quarantine, adjusted_repair_audit = (
    reconstruct_adjusted_close(
        cleaned,
        adjustment_factors,
    )
)

valid_adjusted_close = (
    cleaned["adj_close"].notna()
    & np.isfinite(cleaned["adj_close"])
    & cleaned["adj_close"].gt(0)
)

assert valid_adjusted_close.all()

print(
    "Adjusted closes reconstructed:",
    cleaned["adj_close_reconstructed"].sum(),
)
print(
    "Adjusted-close rows quarantined:",
    len(adjusted_quarantine),
)

display(adjusted_repair_audit.head())

Adjusted closes reconstructed: 28791
Adjusted-close rows quarantined: 0


,ticker,ts,close,original_adj_close,reference_adjustment_factor,adj_close,adj_close_repair_method
0,AAPL,1981-01-08,0.13505,NaN,0.770826,0.104100,exact_reference_ticker_date_factor
1,AAPL,1981-01-13,0.13616,NaN,0.770858,0.104960,exact_reference_ticker_date_factor
2,AAPL,1981-02-03,0.12333,1.264587,0.770853,0.095069,inconsistent_value_rebuilt_with_exact_referenc...
3,AAPL,1981-02-12,0.11663,NaN,0.770872,0.089907,exact_reference_ticker_date_factor
4,AAPL,1981-04-16,0.11161,NaN,0.770858,0.086035,exact_reference_ticker_date_factor


### Adjusted-close repair result

All 18,374 missing and 10,417 positive but inconsistent adjusted closes
were reconstructed using an exact ticker/date factor from the supplied `ohlc`
table. No factor was carried across dates, and no rows required quarantine.

## 11. Review extreme returns and volume

Extreme raw-close returns are compared with the supplied reference table. A
matching reference price provides evidence that the value was not introduced
by corruption in the homework table, but it does not prove whether the move
came from a split, news, or another market event. Such rows remain flagged.

Observed zero volume is retained. Only missing volume is replaced with zero,
and the repair flag distinguishes an imputed zero from an observed zero.

In [33]:
reference_comparison = reference[
    ["ticker", "ts", "close", "adj_close"]
].copy()

reference_comparison["ticker"] = (
    reference_comparison["ticker"]
    .astype("string")
    .str.replace(r"<[^>]+>", "", regex=True)
    .str.replace(r"\s+", "", regex=True)
    .str.upper()
)

reference_comparison["ts"] = pd.to_datetime(
    reference_comparison["ts"],
    errors="coerce",
)

reference_comparison = reference_comparison.rename(
    columns={
        "close": "reference_close",
        "adj_close": "reference_adj_close",
    }
)

assert not reference_comparison.duplicated(
    key_columns
).any()

In [34]:
extreme_review = (
    diagnostic.loc[
        extreme_return_mask,
        ["ticker", "ts", "close", "simple_return"],
    ]
    .merge(
        reference_comparison,
        on=key_columns,
        how="left",
        validate="one_to_one",
    )
)

extreme_review["close_matches_reference"] = np.isclose(
    extreme_review["close"],
    extreme_review["reference_close"],
    rtol=0,
    atol=1e-10,
)

extreme_review["absolute_return"] = (
    extreme_review["simple_return"].abs()
)

print("Extreme-return rows:", len(extreme_review))
print(
    "Closes matching the reference:",
    extreme_review["close_matches_reference"].sum(),
)

display(
    extreme_review.nlargest(
        10,
        "absolute_return",
    )
)

Extreme-return rows: 5056
Closes matching the reference: 5056


,ticker,ts,close,simple_return,reference_close,reference_adj_close,close_matches_reference,absolute_return
4425,TRV,1976-10-28,9.220000,1.000000,9.220000,3.210000,True,1.000000
4429,TRV,1981-08-10,12.060000,1.000000,12.060000,4.200000,True,1.000000
4427,TRV,1978-12-20,8.030000,0.997512,8.030000,2.800000,True,0.997512
3066,MCD,1968-01-02,0.174900,-0.753620,0.174900,0.075921,True,0.753620
98,AAPL,2000-09-29,0.459820,-0.518695,0.459820,0.388310,True,0.518695
4430,TRV,1981-08-11,6.030000,-0.500000,6.030000,1.570000,True,0.500000
4426,TRV,1976-10-29,4.690000,-0.491323,4.690000,0.997070,True,0.491323
4428,TRV,1978-12-21,4.090000,-0.490660,4.090000,0.924090,True,0.490660
3954,NVDA,2000-03-07,0.173570,0.424106,0.173570,0.159200,True,0.424106
3996,NVDA,2004-08-06,0.078584,-0.352314,0.078584,0.072076,True,0.352314


In [35]:
extreme_keys = (
    extreme_review[key_columns]
    .drop_duplicates()
    .assign(extreme_return_flag=True)
)

cleaned = cleaned.merge(
    extreme_keys,
    on=key_columns,
    how="left",
    validate="one_to_one",
)

cleaned["extreme_return_flag"] = (
    cleaned["extreme_return_flag"]
    .fillna(False)
    .astype(bool)
)

In [36]:
invalid_volume_after_cleaning = (
    cleaned["volume"].isna()
    | ~np.isfinite(cleaned["volume"])
    | cleaned["volume"].lt(0)
)

volume_quarantine = cleaned.loc[
    invalid_volume_after_cleaning
].copy()

volume_quarantine["quarantine_reason"] = (
    "Volume remained missing, negative, or non-finite."
)

cleaned = (
    cleaned.loc[~invalid_volume_after_cleaning]
    .copy()
    .reset_index(drop=True)
)

cleaned["zero_volume_flag"] = cleaned["volume"].eq(0)

print(
    "Observed zero-volume rows:",
    (
        cleaned["zero_volume_flag"]
        & ~cleaned["volume_filled"]
    ).sum(),
)
print(
    "Imputed zero-volume rows:",
    cleaned["volume_filled"].sum(),
)
print(
    "Invalid-volume rows quarantined:",
    len(volume_quarantine),
)

Observed zero-volume rows: 75
Imputed zero-volume rows: 0
Invalid-volume rows quarantined: 0


### Return and volume decision

All 5,056 extreme raw closes matched the exact ticker/date values in the
reference table, so they were retained as review flags rather than treated as
confirmed errors. The data contained 75 observed zero-volume rows; these were
retained and distinguished from imputed zeros. No invalid-volume rows required
quarantine.

## 12. Demonstrate missing-value repairs

The real data contains no missing raw OHLC or volume cells after deduplication.
The following artificial example verifies forward-filling within a ticker,
volume imputation, removal of all-OHLC-missing rows, and quarantine of an
unresolved leading price gap.

In [37]:
repair_demo = pd.DataFrame(
    {
        "ticker": [
            "TEST",
            "TEST",
            "TEST",
            "LEAD",
        ],
        "ts": pd.to_datetime(
            [
                "2024-01-02",
                "2024-01-03",
                "2024-01-04",
                "2024-01-02",
            ]
        ),
        "open": [10.0, np.nan, np.nan, 10.0],
        "high": [11.0, 11.0, np.nan, 11.0],
        "low": [9.0, 9.0, np.nan, 9.0],
        "close": [10.0, 10.5, np.nan, np.nan],
        "adj_close": [10.0, 10.5, np.nan, 10.0],
        "volume": [100.0, np.nan, 100.0, 100.0],
    }
)

display(repair_demo)

,ticker,ts,open,high,low,close,adj_close,volume
0,TEST,2024-01-02,10.0,11.0,9.0,10.0,10.0,100.0
1,TEST,2024-01-03,NaN,11.0,9.0,10.5,10.5,NaN
2,TEST,2024-01-04,NaN,NaN,NaN,NaN,NaN,100.0
3,LEAD,2024-01-02,10.0,11.0,9.0,NaN,10.0,100.0


In [38]:
demo_after_missing, demo_removed = (
    handle_missing_values(repair_demo)
)

demo_repaired, demo_quarantine, demo_repair_audit = (
    repair_invalid_ohlc(demo_after_missing)
)

display(demo_after_missing)
display(demo_removed)
display(demo_quarantine)

,ticker,ts,open,high,low,close,adj_close,volume,raw_price_filled,volume_filled
0,LEAD,2024-01-02,10.0,11.0,9.0,NaN,10.0,100.0,False,False
1,TEST,2024-01-02,10.0,11.0,9.0,10.0,10.0,100.0,False,False
2,TEST,2024-01-03,10.0,11.0,9.0,10.5,10.5,0.0,True,True


,ticker,ts,open,high,low,close,adj_close,volume,removal_reason
0,TEST,2024-01-04,NaN,NaN,NaN,NaN,NaN,100.0,All raw OHLC prices were missing.


,ticker,ts,open,high,low,close,adj_close,volume,raw_price_filled,volume_filled,ohlc_repaired,close_repaired,ohlc_repair_method,quarantine_reason
0,LEAD,2024-01-02,10.0,11.0,9.0,NaN,10.0,100.0,False,False,False,False,<NA>,Invalid OHLC could not be repaired defensibly.


In [39]:
filled_row = demo_after_missing.loc[
    (demo_after_missing["ticker"] == "TEST")
    & (
        demo_after_missing["ts"]
        == pd.Timestamp("2024-01-03")
    )
].iloc[0]

assert filled_row["open"] == 10.0
assert bool(filled_row["raw_price_filled"])
assert filled_row["volume"] == 0
assert bool(filled_row["volume_filled"])

assert len(demo_removed) == 1
assert len(demo_quarantine) == 1
assert demo_quarantine.iloc[0]["ticker"] == "LEAD"
assert valid_ohlc_mask(demo_repaired).all()

print("Injected repair checks passed.")

Injected repair checks passed.


### Injected repair result

The test confirmed that partial raw-price gaps are filled only within a ticker,
missing volume is replaced with a flagged zero, and rows with all raw prices
missing are removed before filling. A leading close gap with no defensible
repair value was quarantined.

## 13. Row accounting

Every raw row must be retained, removed as an exact duplicate or all-price
missing row, or placed in quarantine. This ensures that no observations
disappear without explanation.

In [40]:
quarantine_frames = [
    ohlc_quarantine,
    adjusted_quarantine,
    volume_quarantine,
]

nonempty_quarantines = [
    frame
    for frame in quarantine_frames
    if not frame.empty
]

if nonempty_quarantines:
    all_quarantined = pd.concat(
        nonempty_quarantines,
        ignore_index=True,
        sort=False,
    )
else:
    all_quarantined = pd.DataFrame()

initial_rows = len(raw)
exact_duplicates_removed = int(
    extra_exact_duplicates.sum()
)
all_raw_missing_removed = len(
    removed_all_raw_missing
)
quarantined_rows = len(all_quarantined)
retained_rows = len(cleaned)

accounted_rows = (
    exact_duplicates_removed
    + all_raw_missing_removed
    + quarantined_rows
    + retained_rows
)

row_accounting = pd.DataFrame(
    {
        "category": [
            "Initial raw rows",
            "Exact duplicates removed",
            "All-OHLC-missing rows removed",
            "Rows quarantined",
            "Rows retained",
        ],
        "row_count": [
            initial_rows,
            exact_duplicates_removed,
            all_raw_missing_removed,
            quarantined_rows,
            retained_rows,
        ],
    }
)

display(row_accounting)

assert accounted_rows == initial_rows
print("Row accounting balances.")

,category,row_count
0,Initial raw rows,397711
1,Exact duplicates removed,29460
2,All-OHLC-missing rows removed,0
3,Rows quarantined,0
4,Rows retained,368251


Row accounting balances.


### Accounting result

All 397,711 raw rows were accounted for. After removing 29,460 exact
duplicates, 368,251 cleaned observations remained. No real-data rows required
removal for all-missing OHLC values or quarantine.

## 14. Final validation

The retained data must have unique keys, valid prices and OHLC relationships,
finite nonnegative volume, and complete repair flags. The same diagnostic
functions are rerun after cleaning.

In [41]:
cleaned = (
    cleaned
    .sort_values(["ticker", "ts"])
    .reset_index(drop=True)
)

flag_columns = [
    "ticker_was_normalized",
    "raw_price_filled",
    "volume_filled",
    "ohlc_repaired",
    "close_repaired",
    "adj_close_reconstructed",
    "extreme_return_flag",
    "zero_volume_flag",
]

# Keys must be complete and unique.
assert cleaned["ticker"].notna().all()
assert cleaned["ts"].notna().all()
assert not cleaned.duplicated(key_columns).any()

# Every retained price must be finite and positive.
assert cleaned[price_columns].notna().all().all()
assert np.isfinite(
    cleaned[price_columns].to_numpy(dtype=float)
).all()
assert cleaned[price_columns].gt(0).all().all()

# Adjusted closes must agree with the exact ticker/date factor.
final_adjustment_check = cleaned.merge(
    adjustment_factors,
    on=["ticker", "ts"],
    how="left",
    validate="one_to_one",
)

expected_adjusted_close = (
    final_adjustment_check["close"]
    * final_adjustment_check["reference_adjustment_factor"]
)

assert np.isfinite(expected_adjusted_close).all()
assert np.isclose(
    final_adjustment_check["adj_close"],
    expected_adjusted_close,
    rtol=0,
    atol=0.01,
).all()

# Both open and close must lie within the daily range.
assert cleaned["low"].le(cleaned["high"]).all()
assert cleaned["open"].between(
    cleaned["low"],
    cleaned["high"],
).all()
assert cleaned["close"].between(
    cleaned["low"],
    cleaned["high"],
).all()

# Volume must be finite and nonnegative.
assert cleaned["volume"].notna().all()
assert np.isfinite(cleaned["volume"]).all()
assert cleaned["volume"].ge(0).all()

# Repair flags must be present for every retained observation.
assert cleaned[flag_columns].notna().all().all()

# Each ticker must remain chronologically ordered.
assert (
    cleaned
    .groupby("ticker")["ts"]
    .apply(lambda dates: dates.is_monotonic_increasing)
    .all()
)

print("All strict validation checks passed.")

All strict validation checks passed.


In [42]:
_, _, market_quality_after = detect_market_issues(
    cleaned
)

missing_summary_after, missing_dates_after = (
    find_missing_trading_days(
        cleaned,
        reference_calendar,
    )
)

quality_report_after = pd.concat(
    [
        market_quality_after,
        missing_dates_after,
    ],
    ignore_index=True,
)

after_issue_counts = (
    quality_report_after
    .groupby(["issue_type", "severity"])
    .size()
    .rename("issue_count")
    .to_frame()
)

display(after_issue_counts)

high_severity_after = (
    quality_report_after["severity"]
    .eq("high")
    .sum()
)

print(
    "High-severity issues after cleaning:",
    high_severity_after,
)
print(
    "Missing trading dates after cleaning:",
    len(missing_dates_after),
)

assert high_severity_after == 0
assert len(missing_dates_after) == 0

,,issue_count
issue_type,severity,
extreme_return,medium,5056
zero_volume,low,75


High-severity issues after cleaning: 0
Missing trading dates after cleaning: 0


## 15. Before and after comparison

The comparison summarizes how cleaning changed completeness and validity.
Repair counts provide an audit of every modification.

In [43]:
summary_columns = price_columns + ["volume"]
summary_statistics = [
    "count",
    "mean",
    "std",
    "min",
    "max",
]

before_statistics = (
    raw[summary_columns]
    .describe()
    .T[summary_statistics]
    .add_prefix("before_")
)

after_statistics = (
    cleaned[summary_columns]
    .describe()
    .T[summary_statistics]
    .add_prefix("after_")
)

before_after_statistics = before_statistics.join(
    after_statistics
)

display(before_after_statistics)

,before_count,before_mean,before_std,before_min,before_max,after_count,after_mean,after_std,after_min,after_max
open,397711.0,4.628548e+01,6.713259e+01,0.000000,6.042600e+02,368251.0,4.630878e+01,6.714300e+01,0.005208,6.042600e+02
high,397711.0,4.674766e+01,6.777821e+01,0.005208,6.086300e+02,368251.0,4.677120e+01,6.778907e+01,0.005208,6.086300e+02
low,397711.0,4.581628e+01,6.647718e+01,0.004801,5.978800e+02,368251.0,4.583977e+01,6.648920e+01,0.004801,5.978800e+02
close,397711.0,4.629290e+01,6.714188e+01,0.005208,6.054000e+02,368251.0,4.631671e+01,6.715469e+01,0.005208,6.054000e+02
adj_close,377856.0,5.165225e+01,1.945840e+02,0.002945,1.044251e+04,368251.0,3.727417e+01,6.358759e+01,0.002945,6.054000e+02
volume,397711.0,3.097773e+07,1.266237e+08,0.000000,9.276606e+09,368251.0,3.095142e+07,1.266145e+08,0.000000,9.276606e+09


In [44]:
repair_summary = pd.Series(
    {
        "exact_duplicates_removed": exact_duplicates_removed,
        "all_raw_missing_removed": all_raw_missing_removed,
        "raw_price_rows_filled": int(
            cleaned["raw_price_filled"].sum()
        ),
        "volume_rows_filled": int(
            cleaned["volume_filled"].sum()
        ),
        "ohlc_rows_repaired": int(
            cleaned["ohlc_repaired"].sum()
        ),
        "adjusted_closes_reconstructed": int(
            cleaned["adj_close_reconstructed"].sum()
        ),
        "extreme_returns_retained": int(
            cleaned["extreme_return_flag"].sum()
        ),
        "observed_zero_volume_retained": int(
            cleaned["zero_volume_flag"].sum()
        ),
        "rows_quarantined": quarantined_rows,
        "final_rows": len(cleaned),
    },
    name="row_count",
)

display(repair_summary.to_frame())

,row_count
exact_duplicates_removed,29460
all_raw_missing_removed,0
raw_price_rows_filled,0
volume_rows_filled,0
ohlc_rows_repaired,5
adjusted_closes_reconstructed,28791
extreme_returns_retained,5056
observed_zero_volume_retained,75
rows_quarantined,0
final_rows,368251


### Final validation result

All high-severity issues were resolved. The cleaned data has unique ticker/date
keys, positive finite prices, valid OHLC bounds, and finite nonnegative volume.
The 5,056 reference-consistent extreme returns and 75 observed zero-volume rows
remain as documented review flags.

## 16. Save and verify the cleaned table

Dates are stored as ISO `YYYY-MM-DD` strings so later SQL date filters work
consistently. The cleaned data is written to a new `ohlc_hw_cleaned` table in
the local database copy. The original `ohlc_hw` table is checked before and
after writing.

In [45]:
cleaned_for_database = cleaned.copy()

cleaned_for_database["ts"] = (
    pd.to_datetime(
        cleaned_for_database["ts"],
        errors="raise",
    )
    .dt.strftime("%Y-%m-%d")
)

assert cleaned_for_database["ts"].str.fullmatch(
    r"\d{4}-\d{2}-\d{2}"
).all()

print("Rows ready to save:", len(cleaned_for_database))
print("Columns ready to save:")
print(cleaned_for_database.columns.tolist())

Rows ready to save: 368251
Columns ready to save:
['ts', 'open', 'high', 'low', 'close', 'adj_close', 'volume', 'ticker', 'original_ticker', 'ticker_was_normalized', 'raw_price_filled', 'volume_filled', 'ohlc_repaired', 'adj_close_reconstructed', 'close_repaired', 'ohlc_repair_method', 'adj_close_repair_method', 'extreme_return_flag', 'zero_volume_flag']


In [46]:
raw_row_count_before = len(raw)

raw_fingerprint_before = (
    pd.util.hash_pandas_object(
        raw,
        index=False,
    )
    .sum()
)

In [47]:
with sqlite3.connect(DB_PATH) as connection:
    cleaned_for_database.to_sql(
        "ohlc_hw_cleaned",
        connection,
        if_exists="replace",
        index=False,
        chunksize=5000,
    )

    saved_cleaned = pd.read_sql_query(
        """
        SELECT *
        FROM ohlc_hw_cleaned
        ORDER BY ticker, ts
        """,
        connection,
    )

    raw_after_write = pd.read_sql_query(
        "SELECT * FROM ohlc_hw",
        connection,
    )

    final_tables = pd.read_sql_query(
        """
        SELECT name
        FROM sqlite_master
        WHERE type = 'table'
        ORDER BY name
        """,
        connection,
    )

In [48]:
assert len(saved_cleaned) == len(
    cleaned_for_database
)

assert not saved_cleaned.duplicated(
    ["ticker", "ts"]
).any()

assert saved_cleaned["ts"].str.fullmatch(
    r"\d{4}-\d{2}-\d{2}"
).all()

assert saved_cleaned[price_columns].notna().all().all()
assert np.isfinite(
    saved_cleaned[price_columns].to_numpy(dtype=float)
).all()
assert saved_cleaned[price_columns].gt(0).all().all()

assert saved_cleaned["open"].between(
    saved_cleaned["low"],
    saved_cleaned["high"],
).all()

assert saved_cleaned["close"].between(
    saved_cleaned["low"],
    saved_cleaned["high"],
).all()

assert np.isfinite(saved_cleaned["volume"]).all()
assert saved_cleaned["volume"].ge(0).all()

raw_fingerprint_after = (
    pd.util.hash_pandas_object(
        raw_after_write,
        index=False,
    )
    .sum()
)

assert len(raw_after_write) == raw_row_count_before
assert raw_fingerprint_after == raw_fingerprint_before

display(final_tables)
print("Saved cleaned rows:", len(saved_cleaned))
print("Original ohlc_hw table remained unchanged.")

,name
0,market_caps
1,ohlc
2,ohlc_hw
3,ohlc_hw_cleaned


Saved cleaned rows: 368251
Original ohlc_hw table remained unchanged.


## Conclusion

The pipeline normalized ticker labels, removed exact duplicates, diagnosed all
required issue types, repaired five invalid opening prices, and reconstructed
28,791 missing or inconsistent adjusted closes from exact reference factors.
All 368,251 retained rows
passed strict validation and were saved to `ohlc_hw_cleaned`. The original
`ohlc_hw` table remained unchanged.